# 07 — Trial Rejection

Detects and removes bad epochs after ICA cleaning. Three steps:
1. **Inspect** — auto-detect bad epochs by peak-to-peak threshold, then confirm interactively
2. **Propagate** — apply the same rejections to all other epoching windows
3. **Apply** — drop the rejected epochs and save final clean epoch files

Rejection thresholds are set in the config under `analysis.artifacts`.

**Requires:** Qt5 backend for interactive plots

**Input:** `<subject>_<window>_clean-epo.fif`  
**Output:** `<subject>_<window>_final-epo.fif`, `<subject>_<window>_rejected_indices.json`

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib
matplotlib.use('Qt5Agg')

from eeg_toolkit import (
    load_config,
    find_subjects,
    inspect_trials_subject,
    propagate_rejections_subject,
    apply_rejection_subject,
    get_clean_epochs_path,
    get_final_epochs_path,
)

# ── Update this path to point to your experiment config ──
cfg = load_config('../configs/your_experiment.yaml')

subjects = find_subjects(cfg)
print(f"Subjects: {len(subjects)}")

In [ ]:
# ── Step 1: Inspect trials (test subject) ──
# Auto-flagged epochs are shown; click to RESCUE (keep), leave unchecked to REJECT.
# Close the window when done.
test_subject = subjects[0]
inspect_window = cfg.analysis.artifacts.inspect_window

print(f"Inspecting: {test_subject} (window: {inspect_window})\n")
inspect_trials_subject(cfg, test_subject, window_name=inspect_window,
                       overwrite=True, verbose=True)

In [ ]:
# ── Step 1 (all subjects): Inspect trials sequentially ──
from eeg_toolkit.artifacts import inspect_all_trials

inspect_summary = inspect_all_trials(cfg, overwrite=False, verbose=True)

In [ ]:
# ── Step 2: Propagate rejections to other windows ──
from eeg_toolkit.artifacts import propagate_rejections_all

prop_summary = propagate_rejections_all(cfg, overwrite=False, verbose=True)

In [ ]:
# ── Step 3: Apply rejections and save final epoch files ──
from eeg_toolkit.artifacts import apply_rejection_all

apply_summary = apply_rejection_all(cfg, overwrite=False, verbose=True)

In [ ]:
# ── Verification: final trial counts per subject ──
import mne
from eeg_toolkit import find_subjects, get_final_epochs_path

subjects = find_subjects(cfg)
windows  = [w.name for w in cfg.epoching_windows]

print(f"=== Final epoch counts ({len(subjects)} subjects) ===\n")
for subj in subjects:
    counts = {}
    for w in windows:
        path = get_final_epochs_path(cfg, subj, w)
        if path.exists():
            epo = mne.read_epochs(path, preload=False, verbose='WARNING')
            counts[w] = len(epo)
        else:
            counts[w] = 'MISSING'
    count_str = ', '.join(f"{w}={n}" for w, n in counts.items())
    print(f"  {subj}: {count_str}")